# Obtaining the cutouts of the plates

In [1]:
import json
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import random
import base64
import gzip
import requests
import os

##

In [12]:
index = {}

with open("../Step_2_Integration/observability_bright.json", "rb") as f:
    while True:
        pos = f.tell()
        line = f.readline()

        if not line:
            break

        s = line.strip()

        if s in (b"", b"{", b"}"):
            continue

        # key is before first colon
        key_bytes = s.split(b":", 1)[0]
        key = json.loads(key_bytes.decode())

        index[int(key)] = pos

print("nkeys:", len(index))

def load_observability_key(filename, index, key):
    key = int(key)

    with open(filename, "rb") as f:
        f.seek(index[key])
        line = f.readline().decode().strip()

    if line.endswith(","):
        line = line[:-1]

    rec = json.loads("{" + line + "}")
    data = np.asarray(rec[str(key)])
    return data

nkeys: 54355


In [13]:
def retrieve_and_save(plate_id, sol_id, ra, dec, path, mpcnum):
    os.makedirs(path, exist_ok=True)

    url = "https://api.starglass.cfa.harvard.edu/public/dasch/dr7/cutout"
    payload = {
        "plate_id": plate_id,
        "solution_number": int(sol_id),
        "center_ra_deg": float(ra),
        "center_dec_deg": float(dec),
    }

    r = requests.post(url, json=payload, headers={"Accept": "application/json"})
    r.raise_for_status()

    fits_bytes = gzip.decompress(base64.b64decode(r.json()))

    filename = f"{mpcnum}_{plate_id}.fits"
    filepath = os.path.join(path, filename)

    with open(filepath, "wb") as f:
        f.write(fits_bytes)

    return filepath

In [14]:
def get_all_files(mpcnum,cutoff=None):
    data = load_observability_key("../Step_2_Integration/observability_bright.json", index, mpcnum)
    ra,dec,rearth,rsun,dradt,ddecdt,jd,platid,vmag = data.T
    ra = data[:,0].astype(float)
    dec = data[:,1].astype(float)
    rearth = data[:,2].astype(float)
    rsun = data[:,3].astype(float)
    dradt = data[:,4].astype(float)
    ddecdt = data[:,5].astype(float)
    jd = data[:,6].astype(float)
    plate_id, sol_id = np.array([entry.split(':') for entry in data[:,7]]).T
    vmag = data[:,8].astype(float)
    
    if cutoff== None:
        cutoff = len(plate_id)

    print(mpcnum)
    for i in range(cutoff):
        filename = f"{mpcnum}_{plate_id[i]}.fits"
        filepath = os.path.join(str(mpcnum),filename,)

        if os.path.exists(filepath):
            print(f"Skipping {i}: {filename}")
            continue

        print(mpcnum,i)
        retrieve_and_save(plate_id[i],sol_id[i],ra[i],dec[i],str(mpcnum),str(mpcnum))

In [ ]:
mpc_ids = [1, 29, 196, 702, 420, 624, 1437, 15440, 15136, 10886]
for num in mpc_ids:
    get_all_files(num,10)

1
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
29
29 0
29 1
29 2
29 3
29 4
29 5
29 6
29 7
29 8
29 9
196
196 0
196 1
196 2
196 3
196 4
196 5
196 6
196 7
196 8
196 9
702
702 0
702 1
702 2
